In [4]:
# 1 Crear 2000 pedidos usando customer_id
# 2 Crear unas 4500 líneas usando product_id
# 3 Calcular pagos según el total de cada pedido
# 4 Crear reseñas solo para productos entregados
# 5 Validar fechas, importes y claves foráneas (foreign keys)

import pandas as pd
import numpy as np
from faker import Faker
from datetime import timedelta

fake = Faker() # Inicializamos Faker para generar datos fake

np.random.seed(42) # resultados reproducibles


# ---------------------------------------
# FUNCIONES PARA GENERAR DATOS TRANSACCIONALES
# ---------------------------------------

def generate_orders(customers_df, n_orders=2000):
    # Genera pedidos usando customer_id reales de customers_df

    statuses = ["pending", "confirmed", "shipped", "delivered", "cancelled", "returned"]
    status_probs = [0.10, 0.15, 0.20, 0.40, 0.10, 0.05]

    orders = []

    for order_id in range(1, n_orders + 1):
        # Elegimos un cliente existente
        customer_id = np.random.choice(customers_df["customer_id"])

        # Elegimos un estado de pedido
        status = np.random.choice(statuses, p=status_probs)

        # Generamos fecha de pedido
        order_date = fake.date_between(start_date="-2y", end_date="today")

        # Inicializamos fechas vacías
        shipped_date = pd.NaT
        delivered_date = pd.NaT

        # Si el pedido fue enviado, entregado o devuelto, debe tener fecha de envío
        if status in ["shipped", "delivered", "returned"]:
            shipped_date = order_date + timedelta(days=np.random.randint(1, 5))

        # Si el pedido fue entregado o devuelto, debe tener fecha de entrega
        if status in ["delivered", "returned"]:
            delivered_date = shipped_date + timedelta(days=np.random.randint(1, 8))

        orders.append({
            "order_id": order_id,
            "customer_id": customer_id,
            "order_date": order_date,
            "shipped_date": shipped_date,
            "delivered_date": delivered_date,
            "status": status
        })

    return pd.DataFrame(orders)


def generate_order_items(orders_df, valid_products, target_lines=4500):
    # Genera líneas de pedido usando order_id reales y product_id válidos
    # Garantiza que cada pedido tenga al menos una línea

    order_items = []
    order_item_id = 1

    # 1 línea obligatoria por cada pedido
    for _, order in orders_df.iterrows():
        product = valid_products.sample(n=1).iloc[0]

        quantity = np.random.randint(1, 5)
        unit_price = float(product["price"])
        discount_rate = np.random.choice([0, 0.05, 0.10], p=[0.75, 0.15, 0.10])
        discount_amount = round(quantity * unit_price * discount_rate, 2)

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": order["order_id"],
            "product_id": product["product_id"],
            "quantity": quantity,
            "unit_price": round(unit_price, 2),
            "discount_amount": discount_amount
        })

        order_item_id += 1

    # Líneas extra hasta llegar aproximadamente a 4500
    extra_lines = target_lines - len(order_items)

    for _ in range(extra_lines):
        order = orders_df.sample(n=1).iloc[0]
        product = valid_products.sample(n=1).iloc[0]

        quantity = np.random.randint(1, 5)
        unit_price = float(product["price"])
        discount_rate = np.random.choice([0, 0.05, 0.10], p=[0.75, 0.15, 0.10])
        discount_amount = round(quantity * unit_price * discount_rate, 2)

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": order["order_id"],
            "product_id": product["product_id"],
            "quantity": quantity,
            "unit_price": round(unit_price, 2),
            "discount_amount": discount_amount
        })

        order_item_id += 1

    return pd.DataFrame(order_items)


def generate_payments(orders_df, order_items_df):
    # Genera pagos coherentes con el total de cada pedido

    payments = []
    payment_id = 1

    payment_methods = ["credit_card", "paypal", "bank_transfer"]

    # Calculamos el total de cada pedido usando sus líneas
    order_totals = (
        order_items_df
        .assign(total=lambda x: x["quantity"] * x["unit_price"] - x["discount_amount"])
        .groupby("order_id")["total"]
        .sum()
        .reset_index()
    )

    for _, order in orders_df.iterrows():
        order_id = order["order_id"]
        status = order["status"]

        # Obtenemos el total del pedido
        total = order_totals.loc[
            order_totals["order_id"] == order_id,
            "total"
        ].iloc[0]

        # Definimos el estado del pago según el estado del pedido
        if status in ["confirmed", "shipped", "delivered"]:
            payment_status = "completed"
            payment_type = "payment"

        elif status == "pending":
            payment_status = np.random.choice(["pending", "failed"], p=[0.7, 0.3])
            payment_type = "payment"

        elif status == "cancelled":
            payment_status = np.random.choice(["failed", "refunded"], p=[0.6, 0.4])
            payment_type = "refund" if payment_status == "refunded" else "payment"

        elif status == "returned":
            payment_status = "refunded"
            payment_type = "refund"

        payments.append({
            "payment_id": payment_id,
            "order_id": order_id,
            "payment_date": order["order_date"] + timedelta(days=np.random.randint(0, 3)),
            "payment_method": np.random.choice(payment_methods),
            "payment_type": payment_type,
            "amount": round(total, 2),
            "status": payment_status
        })

        payment_id += 1

    return pd.DataFrame(payments)


def generate_reviews(orders_df, order_items_df, review_rate=0.35):
    # Genera reseñas solo para líneas de pedidos entregados

    reviews = []

    # Filtramos pedidos entregados
    delivered_orders = orders_df[
        orders_df["status"] == "delivered"
    ][["order_id", "delivered_date"]]

    # Obtenemos líneas de pedido que pertenecen a pedidos entregados
    eligible_items = order_items_df.merge(delivered_orders, on="order_id")

    # Seleccionamos aproximadamente el 35% de esas líneas
    review_sample = eligible_items.sample(
        frac=review_rate,
        random_state=42
    )

    for review_id, (_, row) in enumerate(review_sample.iterrows(), start=1):
        reviews.append({
            "review_id": review_id,
            "order_item_id": row["order_item_id"],
            "review_date": row["delivered_date"] + timedelta(days=np.random.randint(1, 30)),
            "rating": np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.10, 0.20, 0.35, 0.30]),
            "comment": fake.sentence()
        })

    return pd.DataFrame(reviews)


# ---------------------------------------
# CARGA DE DATOS DE ENG A
# ---------------------------------------

customers_df = pd.read_csv("customers.csv")

products_df = pd.read_csv("products.csv")


# ---------------------------------------
# VALIDACIONES DE DATOS MAESTROS
# ---------------------------------------

assert customers_df["customer_id"].is_unique # Verificamos que no haya IDs duplicados en clientes

assert products_df["product_id"].is_unique # Verificamos que no haya IDs duplicados en productos

assert len(customers_df) >= 500 # al menos 500 clientes

assert len(products_df) >= 70 # al menos 70 productos

assert (products_df["stock"] >= 0).all() # no stock negativo


# ---------------------------------------
# FILTRAR PRODUCTOS VÁLIDOS
# ---------------------------------------

# Solo usamos productos:
# - disponibles (is_available = True)
# - con stock positivo
valid_products = products_df[
    (products_df["is_available"] == True) &
    (products_df["stock"] > 0)
]


# ---------------------------------------
# GENERACIÓN DE DATOS TRANSACCIONALES
# ---------------------------------------

# 1 Generamos la tabla de pedidos (orders)
# - Se crean 2000 pedidos
# - Cada pedido tendrá un customer_id existente
orders_df = generate_orders(customers_df, n_orders=2000)


# 2 Generamos líneas de pedido (order_items)
# - Usa pedidos ya creados
# - Usa productos válidos
# - Genera aprox 4500 líneas
# - Cada línea conecta order_id con product_id
order_items_df = generate_order_items(
    orders_df,
    valid_products,
    target_lines=4500
)


# 3 Generamos pagos (payments)
# - Cada pedido tiene uno o más pagos
# - El importe debe coincidir con la suma de sus líneas
payments_df = generate_payments(
    orders_df,
    order_items_df
)


# 4 Generamos reseñas (reviews)
# - Solo para pedidos entregados
# - Aproximadamente el 35% de las líneas
# - Cada review se asocia a order_item_id (no a order completo)
reviews_df = generate_reviews(
    orders_df,
    order_items_df,
    review_rate=0.35
)


# ---------------------------------------
# 5 VALIDACIONES FINALES
# ---------------------------------------

# 5.1 VALIDACIÓN DE FECHAS
# order_date <= shipped_date <= delivered_date

# Para pedidos con shipped_date, comprobamos que no sea anterior al order_date
assert (
    orders_df.dropna(subset=["shipped_date"])
    .eval("order_date <= shipped_date")
).all()

# Para pedidos con delivered_date, comprobamos que no sea anterior al shipped_date
assert (
    orders_df.dropna(subset=["delivered_date"])
    .eval("shipped_date <= delivered_date")
).all()


# 5.2 VALIDACIÓN DE ESTADOS DE PEDIDOS

# Un pedido pending NO debe tener delivered_date
assert orders_df.loc[
    orders_df["status"] == "pending",
    "delivered_date"
].isna().all()

# Un pedido cancelled NO debe tener delivered_date
assert orders_df.loc[
    orders_df["status"] == "cancelled",
    "delivered_date"
].isna().all()


# 5.3 VALIDACIÓN DE IMPORTES DE PAGOS

# Calculamos el total de cada pedido desde order_items
order_totals = (
    order_items_df
    .assign(total=lambda x: x["quantity"] * x["unit_price"] - x["discount_amount"])
    .groupby("order_id")["total"]
    .sum()
    .reset_index()
)

# Unimos con payments para comparar
payment_check = payments_df.merge(order_totals, on="order_id")

# Calculamos diferencia entre pago y total del pedido
payment_check["diff"] = abs(payment_check["amount"] - payment_check["total"])

# La diferencia debe ser prácticamente 0
assert (payment_check["diff"] < 0.01).all()


# 5.4 VALIDACIÓN DE CLAVES FORÁNEAS (foreign keys)

# orders.customer_id debe existir en customers
assert orders_df["customer_id"].isin(customers_df["customer_id"]).all()

# order_items.order_id debe existir en orders
assert order_items_df["order_id"].isin(orders_df["order_id"]).all()

# order_items.product_id debe existir en products
assert order_items_df["product_id"].isin(products_df["product_id"]).all()

# payments.order_id debe existir en orders
assert payments_df["order_id"].isin(orders_df["order_id"]).all()

# reviews.order_item_id debe existir en order_items
assert reviews_df["order_item_id"].isin(order_items_df["order_item_id"]).all()


# 5.5 VALIDACIÓN DE REVIEWS SOLO EN PEDIDOS ENTREGADOS

review_check = (
    reviews_df
    .merge(order_items_df, on="order_item_id")
    .merge(orders_df, on="order_id")
)

# Todas las reviews deben pertenecer a pedidos con status = delivered
assert (review_check["status"] == "delivered").all()


# ---------------------------------------
# EXPORTAR CSV
# ---------------------------------------

orders_df.to_csv("orders.csv", index=False)
order_items_df.to_csv("order_items.csv", index=False)
payments_df.to_csv("payments.csv", index=False)
reviews_df.to_csv("reviews.csv", index=False)


# ---------------------------------------
# RESUMEN FINAL
# ---------------------------------------

print("Datos transaccionales generados correctamente")
print("orders:", len(orders_df))
print("order_items:", len(order_items_df))
print("payments:", len(payments_df))
print("reviews:", len(reviews_df))

Datos transaccionales generados correctamente
orders: 2000
order_items: 4500
payments: 2000
reviews: 641
